# Tree-Ring 主表 baseline 单独真实 GPU 链路入口

该 Notebook 只运行 `tree_ring` 的 SD3.5 common-backbone method-faithful 主表轨道。三个 baseline 可在独立 Colab GPU 会话中并行执行, 结果通过各自 transfer manifest 进入 CPU 论文闭合。

Notebook 仅调用 repository workflow 与打包入口, 不承载 baseline 算法、阈值、攻击或统计实现。

## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 配置可访问 `stabilityai/stable-diffusion-3.5-medium` 的 `HF_TOKEN`。
3. 选择 `probe_paper`、`pilot_paper` 或 `full_paper` 运行层级。
4. 本入口生成 `split_observations/tree_ring_baseline_observations.json` 与对应 transfer manifest。
5. 结果包只有在真实执行、fixed-FPR 阈值和完整攻击集合全部通过时才会生成。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME
SLM_WM_PRIMARY_BASELINE_ID = "tree_ring"
os.environ["SLM_WM_PRIMARY_BASELINE_ID"] = SLM_WM_PRIMARY_BASELINE_ID


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub open_clip_torch scikit-learn scipy pandas datasets tqdm


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="external_baseline_method_faithful",
    baseline_id=os.environ["SLM_WM_PRIMARY_BASELINE_ID"],
)
print(paper_run_environment)

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("external_baseline_method_faithful")
print(dependency_report)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from paper_workflow.notebook_utils.notebook_entrypoint import run_workflow

external_baseline_method_faithful_summary = run_workflow(root='.', workflow_name="external_baseline_method_faithful")
external_baseline_method_faithful_summary


In [ ]:
from pathlib import Path

baseline_id = "tree_ring"
run_root = Path('outputs/external_baseline_method_faithful/run_records') / baseline_id
summary_path = run_root / f'{baseline_id}_summary.json'
transfer_path = Path('outputs/external_baseline_method_faithful/split_observations') / f'{baseline_id}_baseline_transfer_manifest.json'
print(summary_path.read_text(encoding='utf-8'))
print(transfer_path.read_text(encoding='utf-8'))


In [ ]:
import os

from paper_workflow.notebook_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(
    root='.',
    workflow_name="external_baseline_method_faithful",
    baseline_id=os.environ['SLM_WM_PRIMARY_BASELINE_ID'],
)
archive_record.to_dict()


In [ ]:
assert external_baseline_method_faithful_summary['run_decision'] == 'pass'
assert external_baseline_method_faithful_summary['external_baseline_method_faithful_ready'] is True
assert archive_record.archive_digest == archive_record.drive_archive_digest
print({'baseline_id': baseline_id, 'archive_path': archive_record.archive_path, 'drive_archive_path': archive_record.drive_archive_path})
